In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.sql("select * from employee_detail")
display(df)

empId,name,dept,salary,level
3,Biden,Accounting,25000,null
8,alex,Accounting,50000,null
2,Dave,Accounting,25000,null
11,joe,Accounting,20000,null
12,obama,security,20000,null
5,smith,Sales,40000,null
1,Clark,Sales,34000,null
4,Ava,Sales,20000,null
6,john,IT,20000,null
10,trump,HR,60000,null


In [0]:
df_grpby = df.groupBy("dept","salary")\
          .agg(count("*").alias("cnt")).filter(col("cnt") > 1)
df_grpby.show()

+----------+------+---+
|      dept|salary|cnt|
+----------+------+---+
|Accounting| 25000|  2|
|        IT| 20000|  2|
+----------+------+---+



In [0]:
df_new = df.withColumn("level", when(df.salary>=50000,"senior").when(df.salary>=30000,"Mid").otherwise("junior"))
df_new.show()

+-----+-----+----------+------+------+
|empId| name|      dept|salary| level|
+-----+-----+----------+------+------+
|    3|Biden|Accounting| 25000|junior|
|    8| alex|Accounting| 50000|senior|
|    2| Dave|Accounting| 25000|junior|
|   11|  joe|Accounting| 20000|junior|
|   12|obama|  security| 20000|junior|
|    5|smith|     Sales| 40000|   Mid|
|    1|Clark|     Sales| 34000|   Mid|
|    4|  Ava|     Sales| 20000|junior|
|    6| john|        IT| 20000|junior|
|   10|trump|        HR| 60000|senior|
|   13| bush|        IT| 20000|junior|
|    9|  kim|        HR| 20000|junior|
|    7|  dev|        IT| 70000|senior|
+-----+-----+----------+------+------+



In [0]:
from pyspark.sql.window import Window
df_win = df.withColumn("rank", rank().over(Window.partitionBy("dept").orderBy(df.salary.desc())))\
           .withColumn("dense_rank", dense_rank().over(Window.partitionBy("dept").orderBy(df.salary.desc())))\
           .withColumn("avg_sal", avg(df.salary).over(Window.partitionBy("dept")))\
           .withColumn("cmsm", sum(df.salary).over(Window.partitionBy('dept').orderBy(df.salary.desc()).rowsBetween(Window.unboundedPreceding,Window.currentRow)))
df_win.show()

+-----+-----+----------+------+-----+----+----------+------------------+------+
|empId| name|      dept|salary|level|rank|dense_rank|           avg_sal|  cmsm|
+-----+-----+----------+------+-----+----+----------+------------------+------+
|    8| alex|Accounting| 50000| NULL|   1|         1|           30000.0| 50000|
|    3|Biden|Accounting| 25000| NULL|   2|         2|           30000.0| 75000|
|    2| Dave|Accounting| 25000| NULL|   2|         2|           30000.0|100000|
|   11|  joe|Accounting| 20000| NULL|   4|         3|           30000.0|120000|
|   10|trump|        HR| 60000| NULL|   1|         1|           40000.0| 60000|
|    9|  kim|        HR| 20000| NULL|   2|         2|           40000.0| 80000|
|    7|  dev|        IT| 70000| NULL|   1|         1|36666.666666666664| 70000|
|    6| john|        IT| 20000| NULL|   2|         2|36666.666666666664| 90000|
|   13| bush|        IT| 20000| NULL|   2|         2|36666.666666666664|110000|
|    5|smith|     Sales| 40000| NULL|   

In [0]:
df_filt = df.filter((df.dept == 'Accounting') & (df.salary > 20000))
df_filt.show()

+-----+-----+----------+------+-----+
|empId| name|      dept|salary|level|
+-----+-----+----------+------+-----+
|    3|Biden|Accounting| 25000| NULL|
|    8| alex|Accounting| 50000| NULL|
|    2| Dave|Accounting| 25000| NULL|
+-----+-----+----------+------+-----+



In [0]:
df.printSchema()

root
 |-- empId: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- level: string (nullable = true)



In [0]:
df_cast = df.withColumn('level', df.level.cast("int") )
df_cast.printSchema()

root
 |-- empId: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- level: integer (nullable = true)



In [0]:
df_sort = df.sort(df.dept.desc(), df.salary.asc())
df_sort.show()

+-----+-----+----------+------+-----+
|empId| name|      dept|salary|level|
+-----+-----+----------+------+-----+
|   12|obama|  security| 20000| NULL|
|    4|  Ava|     Sales| 20000| NULL|
|    1|Clark|     Sales| 34000| NULL|
|    5|smith|     Sales| 40000| NULL|
|   13| bush|        IT| 20000| NULL|
|    6| john|        IT| 20000| NULL|
|    7|  dev|        IT| 70000| NULL|
|    9|  kim|        HR| 20000| NULL|
|   10|trump|        HR| 60000| NULL|
|   11|  joe|Accounting| 20000| NULL|
|    3|Biden|Accounting| 25000| NULL|
|    2| Dave|Accounting| 25000| NULL|
|    8| alex|Accounting| 50000| NULL|
+-----+-----+----------+------+-----+



In [0]:
df_dup = df.dropDuplicates(['salary']).where(df.salary < 25000)
df_dup.show()

+-----+----+----------+------+-----+
|empId|name|      dept|salary|level|
+-----+----+----------+------+-----+
|   11| joe|Accounting| 20000| NULL|
+-----+----+----------+------+-----+



In [0]:
df_splt = spark.sql("select * from split_tbl")
df_splt.show()

+------------+---+
|        name|age|
+------------+---+
|modi melania| 65|
| goerge bush| 45|
|    john doe| 25|
+------------+---+



In [0]:
df_splt1 = df_splt.withColumn('name', split(df_splt.name,' ')).select(explode("name").alias("name"), "age")
df_splt1.show()


+-------+---+
|   name|age|
+-------+---+
|   modi| 65|
|melania| 65|
| goerge| 45|
|   bush| 45|
|   john| 25|
|    doe| 25|
+-------+---+



In [0]:
df_splt2 = df_splt1.groupby("age").pivot("name").sum("age")
df_splt2.show()

+---+----+----+------+----+-------+----+
|age|bush| doe|goerge|john|melania|modi|
+---+----+----+------+----+-------+----+
| 65|NULL|NULL|  NULL|NULL|     65|  65|
| 45|  45|NULL|    45|NULL|   NULL|NULL|
| 25|NULL|  25|  NULL|  25|   NULL|NULL|
+---+----+----+------+----+-------+----+



In [0]:
data = [("John Doe", "john@example.com", 50000.0),
    ("Jane Smith", "jane@example.com", 60000.0),
    ("Bob Johnson", "bob@example.com", 55000.0)]


schema="Name string,email string,salary double"
df=spark.createDataFrame(data,schema)

df1 = df.agg(sum(col("salary")).alias("total_sal"))
display(df1)

total_sal
165000.0


In [0]:
data=[(1,'Watson',34),(1,'Watson',40),(1,'Watson',34),(2,'Alex',45),(2,'Alex',50)]
schema="ID int,Name string,Marks int"
df=spark.createDataFrame(data,schema)
display(df)
df1 = df.groupby("Name","ID").agg(collect_list("Marks"))
display(df1)
# from pyspark.sql.functions import collect_list,collect_set,col

# df_final=df.groupBy(col("ID"),col("Name")).agg(collect_list(col('Marks')))
# display(df_final)

ID,Name,Marks
1,Watson,34
1,Watson,40
1,Watson,34
2,Alex,45
2,Alex,50


Name,ID,collect_list(Marks)
Watson,1,"List(34, 40, 34)"
Alex,2,"List(45, 50)"


In [0]:
df1 = df.groupby("Name","ID").agg(collect_list("Marks"))
display(df1)

Name,ID,collect_list(Marks)
Watson,1,"List(34, 40, 34)"
Alex,2,"List(45, 50)"


In [0]:
from pyspark.sql.types import *
schema = StructType([
  StructField("ProductCode", StringType(), True),
  StructField("Quantity", StringType(), True),
  StructField("UnitPrice", StringType(), True),
  StructField("CustomerID", StringType(), True),
])
 

data = [
  ("Q001", 5, 20.0, "C001"),
  ("Q002", 3, 15.5, "C002"),
  ("Q003", 10, 5.99, "C003"),
  ("Q004", 2, 50.0, "C001"),
  ("Q005", "nein", 12.75, "C002"),
]
 
df = spark.createDataFrame(data, schema=schema)
df.show()

df1 = df.filter(col("Quantity").rlike('^[a-zA-Z]*$'))
display(df1)

+-----------+--------+---------+----------+
|ProductCode|Quantity|UnitPrice|CustomerID|
+-----------+--------+---------+----------+
|       Q001|       5|     20.0|      C001|
|       Q002|       3|     15.5|      C002|
|       Q003|      10|     5.99|      C003|
|       Q004|       2|     50.0|      C001|
|       Q005|    nein|    12.75|      C002|
+-----------+--------+---------+----------+



ProductCode,Quantity,UnitPrice,CustomerID
Q005,nein,12.75,C002


In [0]:
data=[('Paris','Polo, Tennis'),('Matt','Golf, Hockey'),('Sam',None)]
schema="Person string,Games string"
df=spark.createDataFrame(data,schema)
display(df)
df1 = df.withColumn("Games", split("Games",",")).select("Person",explode("Games").alias("Games"))
display(df1)

Person,Games
Paris,"Polo, Tennis"
Matt,"Golf, Hockey"
Sam,null


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6535858204792984>, line 5
      3 df=spark.createDataFrame(data,schema)
      4 display(df)
----> 5 df1 = df.withColumn("Games", split("Games",",")).select("Person",explode("Games").alias("Games"))
      6 display(df1)

NameError: name 'split' is not defined

In [0]:
from pyspark.sql.types import *
data=[(2025,1,'2025-01-01'),
      (2025,1,'2025-01-02'),
      (2025,1,'2025-01-03'),
      (2025,1,'2025-01-04'),
      (2025,1,'2025-01-05'),
      (2025,1,'2025-01-06'),
      (2025,1,'2025-01-07'),
      (2025,2,'2025-01-08'),
      (2025,2,'2025-01-09'),
      (2025,2,'2025-01-10'),
      (2025,2,'2025-01-11'),
      (2025,2,'2025-01-12'),
      (2025,2,'2025-01-13'),
      (2025,2,'2025-01-14')]

schema=StructType([StructField('year',IntegerType(),True),StructField('week_num',IntegerType(),True),StructField('dates',StringType(),True)])
df=spark.createDataFrame(data,schema)
df.display()

df1 = df.withColumn("dates", to_date("dates"))
df2 = df1.groupBy("year","week_num").agg(min("dates").alias("start_week"), max("dates").alias("end_week"))
df2.display()

year,week_num,dates
2025,1,2025-01-01
2025,1,2025-01-02
2025,1,2025-01-03
2025,1,2025-01-04
2025,1,2025-01-05
2025,1,2025-01-06
2025,1,2025-01-07
2025,2,2025-01-08
2025,2,2025-01-09
2025,2,2025-01-10


year,week_num,start_week,end_week
2025,1,2025-01-01,2025-01-07
2025,2,2025-01-08,2025-01-14


In [0]:
from pyspark.sql.functions import count, when,col
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.window import Window

data = [
 (1, "Shipped", "2025-04-01"),
 (1, "Shipped", "2025-04-02"),
 (2, "Delivered", "2025-04-01"),
 (2, "Delivered", "2025-04-02"),
 (2, "Shipped", "2025-04-03"),
 (3, "Shipped", "2025-04-01"),
 (3, "Delivered", "2025-04-02"),
 (3, "Delivered", "2025-04-03")
]

schema = StructType([
 StructField("order_id", IntegerType(), False),
 StructField("order_status", StringType(), True),
 StructField("order_date", StringType(), True)
])

orders_df = spark.createDataFrame(data, schema)
display(orders_df)
df1 = orders_df.withColumn("cnt_tot", count("*").over(Window.partitionBy("order_id").orderBy("order_id")))\
               .withColumn("cnt", count("*").over(Window.partitionBy("order_id","order_status")) )
display(df1)
df2 = df1.select("order_id","order_status","cnt_tot","cnt").distinct().withColumn("shp_pct", (col("cnt")/col("cnt_tot"))*100).filter(col("order_status") == "Shipped")
display(df2)
df3 = df1.select("order_id","order_status","cnt_tot","cnt").distinct().withColumn("dlvr_pct", (col("cnt")/col("cnt_tot"))*100).filter(col("order_status") == "Delivered")
display(df3)
df_f = df2.join(df3, df2.order_id == df3.order_id, "left").withColumn("dlvr_pct", when(col("dlvr_pct").isNull(),0).otherwise(col("dlvr_pct"))).select(df2.order_id,"shp_pct","dlvr_pct")
display(df_f)
# df1 = orders_df.groupBy("order_id","order_status").agg(count("order_id").alias("cnt"), )
# display(df1)
# df2 = df1.groupBy("order_id").agg(count("order_id").alias("cnt_tot"))
# display(df2)

order_id,order_status,order_date
1,Shipped,2025-04-01
1,Shipped,2025-04-02
2,Delivered,2025-04-01
2,Delivered,2025-04-02
2,Shipped,2025-04-03
3,Shipped,2025-04-01
3,Delivered,2025-04-02
3,Delivered,2025-04-03


order_id,order_status,order_date,cnt_tot,cnt
1,Shipped,2025-04-01,2,2
1,Shipped,2025-04-02,2,2
2,Delivered,2025-04-01,3,2
2,Delivered,2025-04-02,3,2
2,Shipped,2025-04-03,3,1
3,Delivered,2025-04-02,3,2
3,Delivered,2025-04-03,3,2
3,Shipped,2025-04-01,3,1


order_id,order_status,cnt_tot,cnt,shp_pct
1,Shipped,2,2,100.0
2,Shipped,3,1,33.33333333333333
3,Shipped,3,1,33.33333333333333


order_id,order_status,cnt_tot,cnt,dlvr_pct
2,Delivered,3,2,66.66666666666666
3,Delivered,3,2,66.66666666666666


order_id,shp_pct,dlvr_pct
1,100.0,0.0
2,33.33333333333333,66.66666666666666
3,33.33333333333333,66.66666666666666


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType 
from pyspark.sql.functions import *

schema = StructType([
StructField("id", IntegerType(), nullable=False),
StructField("name", StringType(), nullable=False), 
StructField("age", IntegerType(), nullable=False),
StructField("department", StringType(), nullable=False), 
StructField("salary", DoubleType(), nullable=False)
])
data = [
    Row(1, "John", 30, "Sales", 50000.0),
    Row(2, "Alice", 28, "Marketing", 60000.0),
    Row(3, "Bob", 32, "Finance", 55000.0),
    Row(4, "Sarah", 29, "Sales", 52000.0),
    Row(5, "Mike", 31, "Finance", 58000.0)
]
employeeDF = spark.createDataFrame(data, schema)
display(employeeDF)
df1 = employeeDF.groupBy("department").agg(sum("salary").alias("tot_sal")).orderBy(col("tot_sal").desc()).limit(1)
display(df1)

id,name,age,department,salary
1,John,30,Sales,50000.0
2,Alice,28,Marketing,60000.0
3,Bob,32,Finance,55000.0
4,Sarah,29,Sales,52000.0
5,Mike,31,Finance,58000.0


department,tot_sal
Finance,113000.0


In [0]:
flights_data = [(1,'Flight2' , 'Los Angeles' , 'London'),
(1,'Flight1' , 'London' , 'Vatican'),
(1,'Flight3' , 'Vatican' , 'Nantes'),
(2,'Flight1' , 'Houston' , 'Paris'),
(2,'Flight2' , 'Paris' , 'Nice')
]

schema = "cust_id int, flight_id string , origin string , destination string"
flights_df = spark.createDataFrame(flights_data,schema)
display(flights_df)

flights_df.createOrReplaceTempView("flights")

df1 = flights_df.orderBy("cust_id").groupBy("cust_id").agg(first("origin").alias("origin"),last("destination").alias("destination"))
display(df1)

cust_id,flight_id,origin,destination
1,Flight2,Los Angeles,London
1,Flight1,London,Vatican
1,Flight3,Vatican,Nantes
2,Flight1,Houston,Paris
2,Flight2,Paris,Nice


cust_id,origin,destination
1,Los Angeles,Nantes
2,Houston,Nice


In [0]:
%sql
select cust_id,
first(origin) as origin,
last(destination) as destination
from flights group By cust_id

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4831173245189394>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'select cust_id,\nfirst(origin) as origin,\nlast(destination) as destination\nfrom flights group By cust_id\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    20